# optiver-close — Phase 2 on Colab

Reproduces Phase 2 from nothing but the repo and a Kaggle account: clone → download the
competition data → build the fixture → **verify the rebuild against the committed
manifest** → run the test suite → run the Phase 2 script → read the numbers.

Phase 2 keeps the Phase 1 harness byte-for-byte — same folds, same 5-date embargo, same
floor — and changes two things, measured separately: a model class that optimises MAE
directly (LightGBM, `objective="l1"`, hyperparameters fixed a priori and never tuned on a
validation fold) and features with memory (within-auction rolling, cross-sectional,
cross-auction state — all causal, proven by truncation). The headline you are reproducing,
pooled OOF MAE in bps over the same 3,293,068 scored rows, dates 181–480:

| model | features | MAE (bps) | vs zero (bps) | vs zero (%) |
|---|---|---|---|---|
| `lgbm_mem` | 31 (+memory) | **6.25592** | **+0.12926** | **+2.02%** |
| `lgbm_row` | 14 row-wise | 6.28432 | +0.10087 | +1.58% |
| `ridge_mem` | 31 (+memory) | 6.31036 | +0.07482 | +1.17% |
| `ridge` (Phase 1) | 14 row-wise | 6.32235 | +0.06283 | +0.98% |
| `zero` (floor) | — | 6.38518 | 0 | 0 |

`+` is better throughout this notebook: the number is how far *below* predict-zero the
model lands.

**SMOKE numbers are never results.** The committed 40-stock fixture exists so the tests are
green on a fresh clone; this notebook runs the **FULL** preset, and only FULL numbers go in
the log.

**Runtime:** CPU is fine — LightGBM is pinned to 8 threads for determinism (see
`boosted.py`), so more cores do not help and a GPU does nothing. Pick a **high-RAM**
runtime if offered. The script records its own `runtime_seconds` in the report it writes
(391 s on the machine that produced the committed copy; ~2.5× that with `--ablate`) —
read that field rather than trusting a number typed here.

**You need:** a Kaggle account that has **joined** the *Optiver — Trading at the Close*
competition (kaggle.com/competitions/optiver-trading-at-the-close → Late Submission /
"I Understand and Accept"), and an API token (`kaggle.json` from kaggle.com → Settings →
Create New Token). Without having accepted the rules, the download in §2 returns a 403.

## 1. The repo

In [ ]:
import os, subprocess
from pathlib import Path

REPO_USER = "Bromine185"
REPO_NAME = "optiver-close"
REPO = Path("/content") / REPO_NAME

if not REPO.exists():
    token = os.environ.get("GITHUB_TOKEN", "")   # empty string → no auth prefix
    auth  = f"{token}@" if token else ""
    url   = f"https://{auth}github.com/{REPO_USER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "master", url, str(REPO)], check=True)
    del auth, url
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)

%cd {REPO}
!git log --oneline -1

In [ ]:
# Colab ships numpy/pandas/pyarrow/scipy/sklearn/matplotlib and (usually) lightgbm.
# pytest is the usual gap; lightgbm is installed only if it is genuinely absent.
import importlib, subprocess, sys

for module, package in [("pytest", "pytest>=8.0"), ("lightgbm", "lightgbm>=4.0")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

sys.path.insert(0, str(REPO / "src"))
import numpy as np, pandas as pd
import lightgbm
print(f"python {sys.version.split()[0]}  numpy {np.__version__}  pandas {pd.__version__}  "
      f"lightgbm {lightgbm.__version__}")

## 2. The raw data, from Kaggle

`data/raw/` is gitignored (641 MB) and `data/fixtures/train.parquet` (130 MB) is too — what
the repo commits is `manifest.json`, the record a rebuild is checked against, plus the 4 MB
smoke fixture the tests run on. So Colab downloads the competition zip once and rebuilds.

Provide the API token one of two ways:
- **Colab secret** (better): key icon in the left sidebar → add `KAGGLE_USERNAME` and
  `KAGGLE_KEY` (the two fields inside your `kaggle.json`), grant this notebook access;
- **upload**: run the cell and pick your `kaggle.json` when asked.

In [ ]:

import json as _json
import shutil

RAW = REPO / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

if not (RAW / "train.csv").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub"], check=True)
    import kagglehub

    try:
        src = Path(kagglehub.competition_download("optiver-trading-at-the-close"))
    except Exception as e:
        if "403" in str(e) or "Forbidden" in str(e):
            raise SystemExit(
                "403: this Kaggle account has not accepted the competition rules — "
                "visit kaggle.com/competitions/optiver-trading-at-the-close and join first."
            ) from e
        # no credentials found anywhere: interactive login (username + API token, no file)
        kagglehub.login()
        src = Path(kagglehub.competition_download("optiver-trading-at-the-close"))

    # kagglehub caches the files already unzipped; link them into data/raw/ where
    # scripts/build_fixture.py expects them (no 641 MB copy).
    for item in src.iterdir():
        dest = RAW / item.name
        if not dest.exists():
            dest.symlink_to(item)

print(f"train.csv: {(RAW / 'train.csv').stat().st_size / 1e6:.0f} MB")

## 3. Build the fixture, and check the rebuild against the committed manifest

The manifest the build writes is committed, so comparing the rebuilt one against
`git show HEAD:` is the reproduction report. Three classes of difference are expected
and harmless — timestamps / `build_seconds` / library `versions`; the parquet `sha256` /
`bytes`, which move with pyarrow's zstd and not with the data; and **last-ulp
floating-point noise** in the summary statistics, because a different numpy or CPU
reduces 5.2 M-row moments in a different order (observed: excess kurtosis agreeing to
2e-15 relative). So the check is structural, not textual — `optiver.manifest`, which has
its own tests: integers, strings and booleans (row counts, null counts, dtypes, coverage,
the gate's verdict) must match **exactly**; floats to 1e-9 relative, six orders looser
than version noise and six orders tighter than any real content drift. **A single
mismatch means the rebuild did not reproduce the fixture, and nothing downstream of it
should be trusted.**


In [ ]:
!python scripts/build_fixture.py

In [ ]:
# The reproduction report: the rebuilt manifest against the committed one (git HEAD),
# compared STRUCTURALLY by optiver.manifest — ints/strings/bools exact, floats to 1e-9
# relative, build-time / library-version / zstd-byte keys ignored. Same check as
# `python scripts/check_manifest.py`; a text diff of the JSON is not (a different numpy
# reduces 5.2 M-row moments in a different order and moves the last digit of kurtosis).
from optiver import manifest as M

report = M.check(REPO)
print(report.summary())
if not report.ok:
    raise SystemExit("the rebuild did not reproduce the fixture; do not trust anything below this cell")

man = _json.loads((REPO / "data" / "fixtures" / "manifest.json").read_text())
print(f"rows {man['rows']:,}  build {man['build_seconds']}s")


## 4. The test suite

Phase 1's 78 tests plus Phase 2's 20. The new ones make causality executable: every
Phase 2 feature is rebuilt after truncating the frame at bucket *s* of the last date, and
again after cutting at date *d*, and every surviving row must be bit-identical — the live
API's actual information set. A perturbation test covers the case truncation cannot
(scaling date *d*'s targets must not move date *d*'s own state features). With the full
fixture present everything runs; on a bare clone a couple skip.

In [ ]:
!python -m pytest -q

## 5. The 2×2, plus the floor

`run_phase2.py` on the FULL preset. Five arms, so the two Phase 2 changes are measured
separately instead of as one confounded jump:

- `zero` and `ridge` are **replicas** — rerun unchanged, they must agree with
  `phase1_baselines.json` to every printed digit, or the comparison is between two
  harnesses rather than two models;
- `lgbm_row` is the new model on the old 14 columns (model-class gain alone);
- `ridge_mem` is the old model on the new 31 columns (the feature gain a linear model can see);
- `lgbm_mem` is both, and the headline — **named before the run, not min-picked after**.

Success looks like: the replica pair matches Phase 1 to the digit, and the scorecard
reproduces the table at the top of this notebook to reporting precision.

In [ ]:
!python scripts/run_phase2.py --preset FULL

### Optional: the drop-one-family ablations

`--ablate` reruns `lgbm_mem` three more times, each with one Phase 2 family removed
(rolling / cross-sectional / state), on exactly the folds that produced the headline. It
costs roughly **3× the LightGBM time** and is what the log's "importance is not value"
finding rests on — the state family shows 4.3% of gain importance and removing it costs
−0.0002 bps. Skip it if you only wanted the 2×2; the committed report already contains it.

In [ ]:
# !python scripts/run_phase2.py --preset FULL --ablate   # uncomment to run (≈3× the cost)

## 6. Read the report

`reports/phase2_lgbm.json` is the file whose committed copy backs every Phase 2 number in
`RESEARCH.md`. Colab's copy should agree with it to reporting precision. Fold-to-fold MAE
genuinely varies by 1.3 bps (that is the data — `date_id` is anonymised and the regime
behind it unknowable; see "Known limitations" in CLAUDE.md), so compare pooled numbers and
within-fold *differences*, never single folds across runs.

In [ ]:
def report_diff(name: str) -> None:
    p = REPO / "reports" / name
    if not p.exists():
        print(f"{name}: not written (section skipped?)")
        return
    d = subprocess.run(["git", "diff", "--stat", "--", str(p.relative_to(REPO))],
                       capture_output=True, text=True).stdout.strip()
    print(f"{name}: {'matches the committed copy' if not d else d}")

report_diff("phase2_lgbm.json")

rep = _json.loads((REPO / "reports" / "phase2_lgbm.json").read_text())
print(f"\npreset {rep['preset']}   runtime {rep['runtime_seconds']:.0f}s   ablate {rep['ablate']}")

print(f"\n{'model':<24s} {'MAE bps':>9s} {'vs zero':>9s} {'%':>7s}")
for row in sorted(rep["scorecard"], key=lambda r: r["mae_bps"]):
    print(f"{row['model']:<24s} {row['mae_bps']:9.5f} {row['vs_zero_bps']:+9.5f} "
          f"{row['vs_zero_pct']:+6.2f}%")

ft = pd.DataFrame(rep["fold_table"]).set_index("model")
print("\nMAE by fold (bps), paired against zero within fold:")
print(ft.to_string(float_format=lambda x: f"{x:8.4f}"))

c = rep["consistency_lgbm_mem"]
for by in ("date_id", "seconds_in_bucket", "stock_id"):
    print(f"\nlgbm_mem by {by}: better than zero on {c[by]['share_of_groups_better']:.1%} of groups, "
          f"mean {c[by]['mean_improvement_bps']:+.4f} bps, worst {c[by]['worst_bps']:+.4f}, "
          f"best {c[by]['best_bps']:+.4f}")

## 7. Two pictures

Left: MAE through the auction, predict-zero against `lgbm_mem`. The edge is largest at the
open (s = 0), holds through the middle, bumps at s = 300 (the indicative-cross publication)
and again at s = 540 — the final bucket, whose labels are among the unverifiable ones, so
that last point is noted rather than interpreted. Right: `lgbm_mem`'s share of total gain
by feature, mean over folds. Importance is where the trees spent splits, not what the splits
were worth — the ablation table in the log is the second half of that sentence.

In [ ]:
import matplotlib.pyplot as plt

bs = pd.DataFrame(rep["by_seconds"])
imp = pd.Series(rep["importance_mean"]["lgbm_mem"]).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.2, 1]})
axes[0].plot(bs["seconds_in_bucket"], bs["mae_zero"], ls="--", label="predict-zero")
axes[0].plot(bs["seconds_in_bucket"], bs["mae"], label="lgbm_mem")
axes[0].set_xlabel("seconds_in_bucket"); axes[0].set_ylabel("MAE (bps)")
axes[0].set_title("MAE through the auction"); axes[0].legend()

axes[1].barh(imp.index, imp.values)
axes[1].set_xlabel("share of total gain (mean over folds)")
axes[1].set_title("lgbm_mem feature importance")
axes[1].tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.show()

---

### What NOT to conclude from a green run

+2.02% over the floor is a *measurement* of an untuned default configuration, not what
LightGBM "can do" here — a tuned number would need a nested split inside the training
dates and its own log entry. Nothing here was scored through the competition API, so
nothing here is a leaderboard number; `BENCHMARKS.md` is where the comparison to
published results lives, and it compares improvement over predict-zero, not raw MAE
across periods.

Phase 3 (`colab_phase3.ipynb`) puts a neural model beside this one, on the same 31
columns and the same folds, and asks what a blend is worth when its weight is fitted
forward in time.

Nothing in this notebook needs to leave the VM — the fixture and report are rebuilt from
the manifest anywhere. If you want to keep Colab's report anyway:

```python
from google.colab import files
files.download("reports/phase2_lgbm.json")
```